# Lab 2: Congo Returns — Multi-Container ML Processing Pipeline 🐳📦

**"The Matrix is everywhere. It is all around us."** — Morpheus, *The Matrix (1999)*

## Welcome to Congo!

You've just joined **Congo**, a scrappy e-commerce startup specializing in consumer electronics. Your first project? Building the ML pipeline for Congo's new automated returns processing system.

Congo has rolled out automated unboxing stations at their warehouses that photograph every returned product. These photos need to be classified to route returns to the correct specialized department:

| Category | Example Products | Routing Destination |
|----------|-----------------|---------------------|
| Computing | laptop, monitor, keyboard, mouse | Refurbishment Center |
| Mobile | smartphone, smartwatch | Carrier Partners |
| Audio | headphones, speaker | Quality Testing Lab |
| Accessories | charger, game controller | Quick Resale |

Each category has different inspection requirements, resale channels, and processing workflows—so accurate classification directly impacts operational efficiency.

**Your Mission**: Build a containerized ML pipeline that watches for incoming product photos, classifies them, and logs the results.


---

## The Architecture

In production, Congo's system spans two datacenters:

- **Warehouse Datacenter**: Where photos are captured and preprocessed
- **Main Datacenter**: Where ML inference happens (GPU resources are expensive!)

The containers communicate over HTTP, just like any microservices architecture.

**For this lab, you'll simulate this architecture on your laptop.** Both containers run locally, but they're properly isolated—communicating only through the network and mounted volumes, exactly as they would in production.

```bash
┌─────────────────────────────────────────────────────────────────────┐
│                    Your Laptop (simulating production)              │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   📁 incoming/                      📁 logs/                       │
│   (simulates warehouse             (simulates main                  │
│    storage volume)                  datacenter storage)             │
│        │                                 ▲                          │
│        ▼                                 │                          │
│   ┌──────────────┐                ┌─────────────┐                   │
│   │ Preprocessor │  ───────────>  │  Inference  │                   │
│   │  Container   │     HTTP       │     API     │                   │
│   │              │                │  Container  │                   │
│   │ (warehouse   │                │ (main       │                   │
│   │  datacenter) │                │  datacenter)│                   │
│   └──────────────┘                └─────────────┘                   │
│                                         │                           │
│                                    port 8000                        │
└─────────────────────────────────────────────────────────────────────┘
```

---

## Setup: Get the Starter Code

The starter code is in the `LAB02_submission/` folder:

```bash
LAB02_submission/
├── inference_api/
│   ├── app.py              # FastAPI app with /predict endpoint
│   ├── requirements.txt    
│   └── imagenet_classes.json
├── preprocessor/
│   ├── watcher.py          # Folder watcher and batch sender
│   └── requirements.txt    
├── warehouse_images/       # product images from the warehouse for testing
├── customer_images/        # product images taken by customers for testing 
├── Readme.md               # Stub for you to fill in
└── Reflection.md           # Stub for you to fill in
```

Also create two directories on your host machine  (wherever you want) for bind mounts:
- `incoming/` — where product photos arrive
- `logs/` — where classification results are stored


**BEFORE YOU START**: Have a look at the deliverables at the end of this notebook. It will give you a clear idea of what you need to build and submit.

---

## Part 1: Build and Test the Inference API
The inference API is the brain of the operation. It accepts images and returns classifications using a pretrained MobileNetV2 model.

### 1.1 Review the Starter Code

Open `inference_api/app.py` and study it carefully. Pay attention to:

1. **How the model is loaded** — When does this happen? Why?
2. **The `/predict` endpoint** — What does it accept? What does it return?
3. **The logging mechanism** — Where are results written? What format?

Also check `requirements.txt` to see what dependencies the API needs.

**Before writing any Docker code**, answer these questions about `app.py`:

1. What port does the FastAPI server need to run on?
2. What path inside the container does the API write logs to?
3. What information is included in each log entry?
4. Are there any other files that `app.py` depends on to run that should be copied into the container (and if so where should they be copied to)?

*Your answers:*

1. 
2. 
3. 
4. 

### 1.2 Write the Inference API Dockerfile

Create a `Dockerfile` in the `inference_api/` folder. Your Dockerfile should:

1. Start from an appropriate base image
2. Set up a working directory for the app (typically `/app`)
3. Install the necessary Python dependencies for `app.py` to run
4. Copy the application code and any other necessary files into the image
5. Configure the image so it will run the FastAPI server

**Think about**:
- What base image makes sense for a Python ML application? (you want something just big enough to run the app, but not much bigger)
- In what order should you copy files to maximize layer caching? (things that change less often should be copied earlier)
- What command runs a FastAPI app with uvicorn? (look at the lecture for examples; read uvicorn documentation if needed)
- If the app runs in a container, what bind address makes it reachable from outside the container?

### 1.3 Build the Inference API Image

Build your Docker image. Choose a meaningful tag name.

If the build fails, read the error messages carefully. Common issues:
- Typos in the Dockerfile
- Missing files that you're trying to COPY
- Incorrect paths

### 1.4 Run the Inference API Container

Run a container from your image. You need to configure:

1. **Port mapping** — Make the API accessible from your host machine
2. **Volume mount** — Mount a `logs/` folder so classification results persist on your host

**Verify it's working**:
- Open `http://localhost:8000/docs` in your browser
- You should see FastAPI's automatic Swagger documentation
- Try the `/predict` endpoint with a test image

### 1.5 Test the API

Upload a test image and verify:
1. The API returns a classification result
2. A log entry appears in `logs/classifications.jsonl` on your host machine

You can test via the Swagger UI, or use the cell below:

In [ ]:
# Test the inference API
import requests
from pathlib import Path

# Update this path to point to a test image
TEST_IMAGE_PATH = "./LAB02_submission/warehouse_images/CUST95991_PROD31293_headphones.png"

url = "http://localhost:8000/predict"

with open(TEST_IMAGE_PATH, "rb") as f:
    files = {"file": (Path(TEST_IMAGE_PATH).name, f, "image/png")}
    response = requests.post(url, files=files)

if response.status_code == 200:
    result = response.json()
    print(f"✅ Classification: {result['top_prediction']}")
    print(f"   Confidence: {result['confidence']}%")
    print(f"\n   Full response: {result}")
else:
    print(f"❌ Error {response.status_code}: {response.text}")

### 1.6 Checkpoint: Inference API Working

Before moving on, verify:
- [ ] Your Dockerfile builds without errors
- [ ] The container runs and the API is accessible at `localhost:8000`
- [ ] You can classify an image via the `/predict` endpoint
- [ ] Log entries appear in `logs/classifications.jsonl` on your host


---

## Part 2: Build and Test the Preprocessor

The preprocessor container watches the `incoming/` folder for new images, extracts metadata from filenames, and sends them to the inference API.

### 2.1 Review the Preprocessor Code

Open `preprocessor/watcher.py` and study it. Pay attention to:

1. **How it monitors the folder** — What library does it use? How often does it check?
2. **Filename parsing** — How does it extract customer_id and product_id from filenames?
3. **API communication** — How does it know where to send requests?
4. **Batching** — Does it send images one at a time or in batches?


Before moving on, answer these questions about `watcher.py`:

1. What environment variable configures the API URL?
2. What happens to images after they're successfully processed?
3. What folder path does the watcher monitor inside the container?


*Your answers:*

1. 
2. 
3. 

### 2.2 Write the Preprocessor Dockerfile

Create a `Dockerfile` in the `preprocessor/` folder. Your Dockerfile should:

1. Start from an appropriate base image
2. Set up a working directory
3. Install the Python dependencies
4. Copy the watcher script
5. Configure the container to run the watcher on startup

**Think about**:
- This container doesn't need PyTorch—what's the minimal set of dependencies?
- What command runs the watcher script?

### 2.3 Build the Preprocessor Image

Build your Docker image with an appropriate tag.

### 2.4 Run the Preprocessor Container

This is where it gets interesting. You need to run the preprocessor so that:

1. **Volume mount** — The `incoming/` folder on your host is accessible inside the container
2. **Environment variable** — The API URL is passed to the container
3. **Networking** — The container can actually reach the inference API

**The key question**: What URL should the preprocessor use to reach the inference API?

Remember:
- Both containers are running on your laptop
- The inference API is mapped to port 8000 on your host
- From inside the preprocessor container, `localhost` refers to *that container*, not your host machine
- On Docker Desktop (Windows/macOS), `host.docker.internal` resolves to your host machine from inside a container

Think about this carefully before proceeding!

### 2.5 Test the Preprocessor

With both containers running:

1. Copy a few test warehouse images into the `incoming/` folder on your host  (hint: you can use a pattern like `CUST2*.png` or `*_laptop.png` to copy only some files.)
2. Watch the preprocessor logs (`docker logs -f <container_name>`)
3. Verify the images get processed
4. Check that results appear in `logs/classifications.jsonl`

### 2.6 Checkpoint: Both Containers Working </font>

Before moving on, verify:
- [ ] Preprocessor Dockerfile builds without errors
- [ ] Preprocessor container can reach the inference API
- [ ] Dropping images in `incoming/` triggers processing
- [ ] Results appear in `logs/classifications.jsonl`



---

## Part 3: Run the Full Pipeline

Now let's process a larger batch of images end-to-end.

### 3.1 Process All Test Images

1. Make sure both containers are running
2. Copy all warehouse test images into `incoming/`
3. Watch the logs to see them being processed
4. Wait for processing to complete

### 3.2 Verify the Results

Check `logs/classifications.jsonl`. Each line should contain:
- Timestamp
- Customer ID (extracted from filename)
- Product ID (extracted from filename)
- Classification result
- Confidence score

In [ ]:
# Analyze the classification results
from pathlib import Path
import pandas as pd

# Update this path to your logs folder
LOG_FILE = Path("path/to/logs/classifications.jsonl")

if not LOG_FILE.exists():
    print(f"Log file not found: {LOG_FILE}")
    print("Update LOG_FILE to point to your actual classifications.jsonl file.")
else:
    # Read JSONL directly into a DataFrame (one JSON object per line)
    df = pd.read_json(LOG_FILE, lines=True)

    if df.empty:
        print("No results found in the log file.")
    else:
        print(f"Total items processed: {len(df)}")
        print("\nSample entry:")
        print(df.iloc[0].to_json(indent=2))

        print("\nTop 10 classifications:")
        print(df["top_prediction"].value_counts().head(10).to_string())

### 3.3 Test Container Independence

Verify that your containers can be managed independently:

1. Stop the preprocessor container
2. Verify the inference API still responds to direct requests
3. Restart the preprocessor
4. Verify it reconnects and continues processing

This independence is crucial in production—you should be able to update or restart one service without affecting the other.

---

## Part 4: Add Required Endpoints

Production APIs need monitoring endpoints. Add these to the inference API.

### 4.1 Add a `/health` Endpoint

Add an endpoint that returns the service health status. This is used by load balancers and orchestration systems to check if the service is ready.

It should return something like:
```json
{
    "status": "healthy",
    "model_loaded": true
}
```

### 4.2 Add a `/stats` Endpoint

Add an endpoint that reads from the log file and returns processing statistics:

- Total items processed
- Breakdown by top classification category
- Average confidence score

**Think about**: The log file is on a mounted volume. How does the API access it?

### 4.3 Rebuild and Test

After modifying `app.py`:

1. Rebuild the inference API image
2. Stop and remove the old container
3. Start a new container from the updated image
4. Test both new endpoints

In [ ]:
# Test the new endpoints
import requests

# Test /health
health_response = requests.get("http://localhost:8000/health")
print("Health check:")
print(health_response.json())

# Test /stats
stats_response = requests.get("http://localhost:8000/stats")
print("\nStatistics:")
print(stats_response.json())

---

## Deliverables

Submit the following (by pushing to your GitHub repo):

1. **`inference_api/Dockerfile`**

2. **`preprocessor/Dockerfile`**

3. **Modified `inference_api/app.py`** — with `/health` and `/stats` endpoints

4. **Readme.md** containing:
   - The exact `docker build` commands you used
   - The exact `docker run` commands you used for each container
   - Brief explanation of how the containers communicate

5. **sample_classifications_20.jsonl** — The first 20 entries from `classifications.jsonl` (see below)

6. **Reflection.md** (1-2 paragraphs) — Answer the question below

7. **This notebook**


To get the first 20 entries from `classifications.jsonl`, you can use the following command in your terminal.

- *On macOS/Linux*:
   ```bash
   head -n 20 logs/classifications.jsonl > sample_classifications_20.jsonl
   ```
- *On Windows PowerShell*:
   ```Powershell
   Get-Content .\logs\classifications.jsonl -TotalCount 20 | Set-Content .\sample_classifications_20.jsonl
   ```

### Reflection Question

In this lab, both containers ran on your laptop. In production, the preprocessor would run in the warehouse datacenter and the inference API would run in Congo's main datacenter.

**How would the architecture and your `docker run` commands differ if these containers were actually running in separate datacenters?**

Consider:
- How would the preprocessor find the inference API?
- What about the shared volumes?
- What new challenges would arise?

*Write your reflection in Reflection.md*

---

## Testing Checklist

Before submitting, verify:

- [ ] Inference API Dockerfile builds without errors
- [ ] Preprocessor Dockerfile builds without errors
- [ ] Inference API container runs and is accessible on port 8000
- [ ] Preprocessor container can reach the inference API
- [ ] Dropping images in `incoming/` triggers processing
- [ ] Results appear in `logs/classifications.jsonl` on your host machine
- [ ] Log entries contain timestamp, customer_id, product_id, and predictions
- [ ] `/health` endpoint returns status
- [ ] `/stats` endpoint returns processing statistics
- [ ] Containers can be stopped and restarted independently

---

## Stretch Goals

If you finish early or want extra practice:

### Basic Stretch Goals
- Add error handling: move failed images to an `errors/` folder
- Implement retry logic in the preprocessor for failed API calls
- Push your images to Docker Hub

### Advanced Stretch Goal: Customer Pre-Return Photo Upload Service

Congo wants customers to upload a photo of their product *before* shipping it back. This enables comparison with the warehouse photo to detect shipping damage.

**Build a third container from scratch** (no starter code!) that:

1. Exposes an API endpoint for customers to upload a photo with their `customer_id` and `product_id`
2. Calls the inference API to classify the product
3. Saves the photo with the naming format: `CUST{customer_id}_PROD{product_id}_prereturn_{classification}.jpg`
4. Returns a confirmation with the classification result

You'll need to:
- Design the API endpoint(s)
- Write all the Python code
- Create the Dockerfile
- Configure networking to reach the inference API
- Set up volume mounting for persistent photo storage

This tests whether you can build a complete containerized service from scratch!

---

## Troubleshooting

Remember, you can always interactively shell into a running container and poke around.

### Container can't reach the API?
- From inside a container, `localhost` refers to *that container*, not your host
- Research how containers can reach services on the host machine
- Check that the API container is running and the port is mapped

### Volume mount not working?
- Make sure the host directory exists before running the container
- Check the path is absolute or relative to your current directory
- On Windows, you may need to enable file sharing in Docker Desktop settings

### Images not being processed?
- Check preprocessor logs: `docker logs <container_name>`
- Verify the image filenames match the expected pattern
- Ensure the API URL environment variable is set correctly

### Build fails?
- Read the error message carefully
- Check for typos in the Dockerfile
- Verify all files being COPYed actually exist

---

## What You Learned

In this lab, you:

1. **Built two Dockerfiles** for different purposes (ML inference vs. preprocessing)
2. **Configured bind mounts** for data persistence across container restarts
3. **Solved container networking** — figured out how containers communicate
4. **Used environment variables** to configure containers at runtime
5. **Managed container lifecycle** — build, run, stop, restart, logs
6. **Simulated a distributed architecture** on a single machine

These are the core skills for deploying ML systems in production. The same patterns apply whether you're running on a laptop, in the cloud, or on a Kubernetes cluster.